# Introduction aux Grands Modèles de Langage (LLMs)

## Qu'est-ce qu'un LLM ?

Un **Grand Modèle de Langage** (Large Language Model) est un modèle d'intelligence artificielle entraîné sur d'immenses quantités de texte. Il est capable de comprendre et de générer du langage naturel de manière cohérente et contextuelle.

### Principes de fonctionnement

- **Architecture Transformer** : introduite en 2017 par Google, elle permet de traiter des séquences de texte en parallèle grâce au mécanisme d'**attention**.
- **Tokens** : le texte est découpé en unités appelées *tokens* (≈ mots ou sous-mots). Les LLMs prédisent le token suivant à chaque étape.
- **Pré-entraînement** : le modèle apprend sur des milliards de documents (livres, articles, code, pages web…).
- **Fine-tuning** : le modèle est ensuite affiné pour suivre des instructions ou adopter un comportement conversationnel (ex : RLHF).

### Principaux fournisseurs de LLMs

| Entreprise | Modèles phares |
|---|---|
| Google | Gemini, Gemma |
| OpenAI | GPT-4o, o1 |
| Anthropic | Claude 3.5 / 4 |
| Meta | LLaMA 4 |
| Mistral AI | Mistral, Mixtral |
| DeepSeek | DeepSeek V4 |

---

**Dans ce workshop**, nous allons :
1. Utiliser l'API **Gemini** (Google) pour des appels dans le cloud
2. Déployer **Gemma 3 1B** localement dans une application web FastAPI

---

## Partie 1 — API Gemini : Configuration et Premier Appel

### Obtenir une clé API Gemini

1. Rendez-vous sur **[Google AI Studio](https://aistudio.google.com/)**
2. Connectez-vous avec votre compte Google
3. Cliquez sur **"Get API key"** → **"Create API key in new project"**
4. Copiez la clé générée (elle commence par `AIza...`)

### Sécuriser la clé API : le fichier `.env`

> ⚠️ **Ne jamais mettre une clé API directement dans le code ou dans un dépôt Git !**

**Bonne pratique** : utiliser un fichier `.env` pour stocker les secrets localement.

**Étape 1** — Créez un fichier `.env` à la racine du projet :
```
GEMINI_API_KEY=AIza...
```

**Étape 2** — Ajoutez `.env` à votre `.gitignore` pour ne jamais l'envoyer sur Git :
```
# .gitignore
.env
flask_app/assets/
```

**Étape 3** — Chargez la clé dans Python avec `python-dotenv` :
```python
from dotenv import load_dotenv
import os

load_dotenv()                          # lit le fichier .env
api_key = os.getenv("GEMINI_API_KEY")  # récupère la variable
```

Le client Gemini lit automatiquement `GEMINI_API_KEY` depuis l'environnement — aucune configuration supplémentaire nécessaire.

In [5]:
# Installation des bibliothèques nécessaires
!pip install google-genai python-dotenv --quiet

from dotenv import load_dotenv
import os

# Chargement des variables depuis le fichier .env
load_dotenv()

api_key ="REDACTED_GEMINI_KEY" or os.environ.get('GEMINI_API_KEY')
if api_key:
    print("Clé API chargée avec succès.")
else:
    print("Clé API introuvable — créez un fichier .env avec GEMINI_API_KEY=votre_clé")

Clé API chargée avec succès.


### Premier appel à l'API Gemini

Le SDK `google-genai` expose un objet `Client` qui gère l'authentification automatiquement. On appelle `client.models.generate_content()` en précisant le modèle et le contenu.

In [8]:
from google import genai

# Le client récupère automatiquement la clé depuis GEMINI_API_KEY
client = genai.Client()

# Fallback si jamais la methode mentionne plus haut ne marche pas
# client = genai.Client(api_key = api_key)

response = client.models.generate_content(
    model="gemini-3-flash-preview", contents="Explain how AI works in a few words"
)
print(response.text)

AI analyzes vast amounts of data to find patterns and make predictions.


---

## Partie 2 Hyperparamètres et Prompt Système

### Quels hyperparamètres peut-on régler ?

| Hyperparamètre | Rôle | Valeur typique |
|---|---|---|
| `temperature` | Créativité vs déterminisme. `0` = réponse prévisible, `2` = très créatif | `0.7` – `1.0` |
| `max_output_tokens` | Longueur maximale de la réponse en tokens | `256` – `2048` |
| `top_p` | *Nucleus sampling* : ne garde que les tokens dont la probabilité cumulée dépasse ce seuil | `0.9` – `0.95` |
| `top_k` | Nombre de tokens candidats pris en compte à chaque étape | `40` – `100` |

### Le Prompt Système

Le **prompt système** (`system_instruction`) est une instruction globale, invisible pour l'utilisateur final, qui définit le comportement global du modèle :

- Son rôle (`"Tu es un expert en finance..."`)  
- Son ton (formel, décontracté, technique)  
- Ses contraintes (`"Réponds uniquement en français"`)  
- Le contexte dans lequel il opère (article, base de données, persona)

```
┌──────────────────────────────────────────────────────┐
│  System Instruction  →  "Tu es un assistant..."      │  ← invisible pour l'utilisateur
├──────────────────────────────────────────────────────┤
│  Message Utilisateur →  "Quelle est la capitale..."  │  ← saisie de l'utilisateur
└──────────────────────────────────────────────────────┘
```

In [10]:
from google import genai
from google.genai import types

# client = genai.Client()

# Effet de la température : basse = prévisible, haute = créatif
question = "Propose-moi une idée de startup innovante en une phrase."

for temp in [0.1, 1.5]:
    print(f"\n--- Température : {temp} ---")
    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=question,
        config=types.GenerateContentConfig(
            temperature=temp,
            max_output_tokens=80,
        )
    )
    print(response.text)


--- Température : 0.1 ---


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
# Exemple de prompt système : changer le ton et le rôle du modèle
response = client.models.generate_content(
    model="gemini-2.5-flash-preview",
    contents="Quelle est la capitale de la France ?",
    config=types.GenerateContentConfig(
        system_instruction=(
            "Tu es un assistant sarcastique qui trouve toutes les questions beaucoup trop évidentes. "
            "Réponds en une phrase avec une touche d'humour."
        ),
        temperature=0.9,
        max_output_tokens=80,
    )
)
print(response.text)

---

## Partie 3 — Exemple Pratique : Questions sur un Article

Nous allons utiliser le **prompt système** pour fournir un article au modèle, puis créer une fonction qui répond aux questions en se basant **uniquement** sur cet article.

C'est une application concrète du pattern **RAG simplifié** (Retrieval-Augmented Generation) : on injecte un contexte dans le prompt système, et le modèle répond à partir de ce contexte uniquement.

In [ ]:
from google import genai
from google.genai import types

client = genai.Client()

ARTICLE = """
Anthropic a conclu un partenariat stratégique avec SpaceX afin d'utiliser l'ensemble de la capacité
informatique du centre de données Colossus 1, situé à Memphis dans le Tennessee. Grâce à cet accord,
l'entreprise spécialisée dans l'intelligence artificielle bénéficiera de plus de 300 mégawatts de
puissance de calcul. Le groupe a également indiqué envisager une collaboration future avec SpaceX
autour du développement d'infrastructures de plusieurs gigawatts dans l'espace.

L'accord doit permettre à Anthropic d'améliorer les performances proposées aux abonnés de ses offres
Claude Pro et Claude Max. Cette alliance intervient pourtant dans un contexte de tensions publiques
entre Anthropic et Elon Musk, propriétaire de SpaceX et fondateur de xAI. Ces derniers mois, le
milliardaire a multiplié les critiques contre Anthropic, notamment sur les questions de régulation de
l'intelligence artificielle, allant jusqu'à accuser l'entreprise de \"détester la civilisation occidentale\".

Parallèlement, xAI poursuit l'expansion rapide de ses infrastructures à Memphis afin de concurrencer
OpenAI, Google et Anthropic sur le marché de l'IA. Le centre Colossus 1 est actuellement son plus
important site de calcul. Le projet fait toutefois face à des critiques locales liées à l'utilisation
de turbines au gaz naturel, accusées d'aggraver la pollution atmosphérique dans la région. Par ailleurs,
la startup fondée en 2021 par d'anciens chercheurs et dirigeants d'OpenAI serait en discussion avec
des investisseurs pour une levée de fonds susceptible de valoriser l'entreprise jusqu'à 900 milliards
de dollars.
"""


def repond_a_parti_de_larticle(text: str) -> str:
    """Répond à une question en se basant uniquement sur l'article fourni."""
    response = client.models.generate_content(
        model="gemini-2.5-flash-preview",
        contents=text,
        config=types.GenerateContentConfig(
            system_instruction=(
                "Tu es un assistant qui répond aux questions en te basant UNIQUEMENT sur l'article "
                "suivant. Si la réponse ne figure pas dans l'article, dis-le clairement.\n\n"
                f"Article :\n{ARTICLE}"
            ),
            temperature=0.2,
            max_output_tokens=500,
        )
    )
    return response.text

In [ ]:
# Test de la fonction avec plusieurs questions
questions = [
    "Quel est l'accord conclu entre Anthropic et SpaceX ?",
    "Quelles tensions sont mentionnées dans l'article ?",
    "À combien est valorisée l'entreprise dans les discussions de levée de fonds ?",
    "Quelle est la couleur préférée d'Elon Musk ?",  # hors article
]

for question in questions:
    print(f"Question : {question}")
    print(f"Réponse  : {repond_a_parti_de_larticle(question)}")
    print("-" * 70)

---

## Partie 4 — Application Web Flask avec un Modèle Local

Nous allons créer une interface web pour interagir avec un modèle IA hébergé **localement** sur votre machine, sans passer par une API externe.

### Structure du projet Flask

```
flask_app/
├── main.py                    ← Application Flask  (flask --app main:app run)
├── templates/
│   └── index.html             ← Interface utilisateur HTML
├── static/
│   └── style.css              ← Styles CSS
└── assets/
    └── gemma-3-1b-it/         ← Poids du modèle (générés dans la dernière cellule)
```

### Fonctionnement de l'application

1. Au démarrage, Flask charge le modèle depuis `assets/gemma-3-1b-it/`
2. La route `/` affiche le formulaire de saisie
3. Lorsque l'utilisateur soumet un prompt, Flask l'envoie au modèle local
4. La réponse générée est affichée dans le panneau **Résultat**

### Lancer l'application

```bash
cd flask_app
pip install flask transformers torch accelerate
flask --app main:app run
```

Puis ouvrez [http://127.0.0.1:5000](http://127.0.0.1:5000) dans votre navigateur.

---

## Partie 5 — Chargement du Modèle Gemma 3 1B avec FastAPI

### Gemma 3 1B — Modèle léger de Google

| Caractéristique | Valeur |
|---|---|
| Paramètres | 1 milliard |
| Fenêtre de contexte | 32 768 tokens |
| Type | Instruction-tuned (`-it`) |
| Fournisseur | Google DeepMind |

### Prérequis : Authentification Hugging Face

Gemma est sous licence restreinte. Vous devez :
1. Créer un compte sur [huggingface.co](https://huggingface.co/)
2. Accepter la licence sur la [page du modèle](https://huggingface.co/google/gemma-3-1b-it)
3. Générer un token d'accès : **Settings → Access Tokens → New token (Read)**

La cellule ci-dessous :
- Vous demande votre token Hugging Face si nécessaire
- Télécharge le modèle depuis le Hub
- Sauvegarde les poids localement dans `fastapi_app/assets/gemma-3-1b-it/`
- Prépare le dossier utilisé ensuite par l'application FastAPI


In [ ]:
!pip install --upgrade transformers huggingface_hub

In [14]:
from transformers import pipeline
from huggingface_hub import login
from dotenv import load_dotenv
import torch
import os

load_dotenv()

hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token)
else:
    print("HF_TOKEN absent : si le modèle est protégé, créez un .env avec HF_TOKEN=hf_...")

SAVE_PATH = "fastapi_app/assets/gemma-3-1b-it"
os.makedirs(SAVE_PATH, exist_ok=True)

device = 0 if torch.cuda.is_available() else -1
torch_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

print("Chargement de google/gemma-3-1b-it depuis Hugging Face...")

pipe = pipeline(
    "text-generation",
    model="google/gemma-3-1b-it",
    device=device,
    torch_dtype=torch_dtype
)

print(f"Sauvegarde dans {SAVE_PATH} ...")
pipe.model.save_pretrained(SAVE_PATH)
pipe.tokenizer.save_pretrained(SAVE_PATH)

print(f"\nModèle sauvegardé dans : {os.path.abspath(SAVE_PATH)}")
print("Lancez l'application FastAPI avec : uvicorn fastapi_app.main:app --reload")


RuntimeError: Failed to import transformers.pipelines because of the following error (look up to see its traceback):
cannot import name 'is_offline_mode' from 'huggingface_hub' (C:\Users\Dany Anderson\.conda\envs\basic\Lib\site-packages\huggingface_hub\__init__.py)